# 01 - Data exploration

Explore the real dataset before training. Prerequisite: the dataset is extracted into `data/` (see `data/README.md`).

This notebook is **unexecuted** (no outputs are stored). Run it from VS Code with the project's `.venv` as the kernel.
Write your own observations in the final markdown cell; do not copy numbers from anywhere else.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import config
from src.data_loader import find_candidate_files, load_dataset

pd.set_option("display.width", 140)

## 1. Which file and which columns?

In [ ]:
for p in find_candidate_files():
    print(p.relative_to(config.PROJECT_ROOT))

data = load_dataset()          # auto-detects the file; honours config.local.json
print("\nFile:", data.path)
print("Mapping (verify against DATA_DICTIONARY.md):")
for role, col in data.mapping.items():
    print(f"  {role:<14} {col}")
data.report

## 2. Shape, sampling interval, time range

In [ ]:
df, mapping = data.df, data.mapping
print("Rows:", len(df), "| columns:", list(df.columns))
print("Sampling interval:", data.interval, "| share of steps at that interval:", data.report["share_of_steps_at_interval"])
print("Range:", df.index.min(), "->", df.index.max())
df.head()

In [ ]:
df.describe().T

## 3. Missing values

In [ ]:
missing = df.isna().mean().mul(100).round(2).rename("% missing")
missing.to_frame()

## 4. Soil-moisture trend and distributions

In [ ]:
sm = mapping["soil_moisture"]
fig, ax = plt.subplots(figsize=(12, 3.5))
df[sm].plot(ax=ax, linewidth=0.4)
df[sm].resample("D").mean().plot(ax=ax, linewidth=1.5)
ax.set_ylabel(sm); ax.set_title("Soil moisture: raw and daily mean"); plt.show()

df.hist(figsize=(12, 3 * ((len(df.columns) + 2) // 3)), bins=40, layout=(-1, 3)); plt.tight_layout(); plt.show()

## 5. Relationships with temperature and rainfall (daily aggregates)

In [ ]:
agg = {c: ("sum" if c == mapping.get("precipitation") else "mean") for c in df.columns}
daily = df.resample("D").agg(agg)
daily.corr().round(2)

In [ ]:
pairs = [(r, mapping[r]) for r in ("temperature", "precipitation") if mapping.get(r)]
fig, axes = plt.subplots(1, max(len(pairs), 1), figsize=(6 * max(len(pairs), 1), 4), squeeze=False)
for ax, (role, col) in zip(axes[0], pairs):
    ax.scatter(daily[col], daily[sm], s=12, alpha=0.6); ax.set_xlabel(col); ax.set_ylabel(sm)
plt.show()

## 6. Your observations (fill in)

- Unit of soil moisture and precipitation (from the data dictionary):
- Sampling interval confirmed / gaps found:
- Sensor dropouts, flatlines or spikes noticed:
- Which sector/subset you chose and why:
- Anything the automatic column mapping got wrong: